In [ ]:
import torch
from torch import nn
import torchvision
from torch.utils import data
from tqdm import tqdm
import numpy as np
from datetime import datetime
import json
import wandb
import os

from models import DINO_CLIP, TwoCropsTransform, GaussianBlur
from datasets import ImageNet, Places365, ArtPlaces, Transforms
from evaluation_utils import evaluate_features

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(torch.cuda.get_device_name())

In [ ]:
LOG_RUN = True

CLIP_MODEL_NAME = "ViT-B/32"
DINO_MODEL_NAME = "dinov2_vitb14"
CONVNEXT_MODEL_NAME = "convnextv2_nano.fcmae_ft_in22k_in1k"
FUSION_HEAD = "LinearSmall"
LAMBDA_ALPHA_REG = 0.1
USE_WEIGHTED_CONCAT = True
USE_DINO_CLS_AND_PATCH_TOKENS = True
USE_PROJ = True
PROJ_DIM = 512
DATASET = "Places365"

BATCH_SIZE = 64
EPOCHS = 1
CRITERION = "cross_entropy"
OPTIMIZER = "adam"
SCHEDULER = "cosine"
SCHEDULER_MIN_LR = None
# match SCHEDULER:
#     case "step":
#         SCHEDULE = [15, 30]
#     case "cosine":
#         SCHEDULE = None
match SCHEDULER:
    case "cosine":
        SCHEDULER_MIN_LR = 0.00001

MOMENTUM = 0.9
WEIGHT_DECAY = 0.00001 # 0.001 1e-4
LR = 0.0001 # 0.001

DIM = 1024 # 512 1024
K = 8192 # 8192 16384 32768 65536
M = 0.999
T = 0.07

EVALUATION_DISTANCE = "l2"
EVALUATION_K = [1, 5]

In [ ]:
config = {
    "clip_model_name": CLIP_MODEL_NAME,
    "dino_model_name": DINO_MODEL_NAME,
    "convnext_model_name": CONVNEXT_MODEL_NAME,
    "fusion_head": FUSION_HEAD,
    "lambda_alpha_reg": LAMBDA_ALPHA_REG,
    "use_weighted_concat": USE_WEIGHTED_CONCAT,
    "use_dino_cls_and_patch_tokens": USE_DINO_CLS_AND_PATCH_TOKENS,
    "use_proj": USE_PROJ,
    "proj_dim": PROJ_DIM,
    "dataset": DATASET,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "criterion": CRITERION,
    "optimizer": OPTIMIZER,
    "optimizer_momentum": MOMENTUM,
    "optimizer_weight_decay": WEIGHT_DECAY,
    "scheduler": SCHEDULER,
    "scheduler_min_lr": SCHEDULER_MIN_LR,
    # "schedule": SCHEDULE,
    "dim": DIM,
    "k": K,
    "m": M,
    "t": T,
}

In [ ]:
if LOG_RUN:
    wandb.login()

In [ ]:
NUM_WORKERS = 8

transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
    torchvision.transforms.RandomApply(
        [torchvision.transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)],
        p=0.8,  # not strengthened
    ),
    torchvision.transforms.RandomGrayscale(p=0.2),
    torchvision.transforms.RandomApply([GaussianBlur([0.1, 2.0])], p=0.5),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# transform = torchvision.transforms.Compose([
#     torchvision.transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
#     torchvision.transforms.RandomGrayscale(p=0.2),
#     torchvision.transforms.ColorJitter(0.4, 0.4, 0.4, 0.4),
#     torchvision.transforms.RandomHorizontalFlip(),
#     torchvision.transforms.ToTensor(),
#     torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = TwoCropsTransform(transform)

match DATASET:
    case "ImageNet":
        dataset = ImageNet(root=r"C:\Users\mariu\Documents\Development\Datasets\ImageNet", transform=transform)
        dataset_val = ImageNet(root=r"C:\Users\mariu\Documents\Development\Datasets\ImageNet", split="val", transform=transform)
    case "Places365":
        dataset = Places365(root=r"C:\Users\mariu\Documents\Development\Datasets\Places365", transform=transform)
        dataset_val = Places365(root=r"C:\Users\mariu\Documents\Development\Datasets\Places365", transform=transform, split="val")
    case "ArtPlaces":
        dataset = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform)
        dataset_val = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280", transform=transform, split="val")

data_loader = data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
data_loader_val = data.DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

### Modell

In [ ]:
model = DINO_CLIP(
    dino_model_name=DINO_MODEL_NAME,
    clip_model_name=CLIP_MODEL_NAME,
    convnext_model_name = CONVNEXT_MODEL_NAME,
    dim = DIM,
    K = K,
    m = M,
    T = T,
    fusion_head=FUSION_HEAD,
    use_weighted_concat = USE_WEIGHTED_CONCAT,
    use_dino_cls_and_patch_tokens = USE_DINO_CLS_AND_PATCH_TOKENS,
    use_proj = USE_PROJ,
    proj_dim = PROJ_DIM
)
model = model.train()
model = model.to(device)

In [ ]:
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name)

In [ ]:
if LOG_RUN:
    config["fusion_head_architecture"] = str(model.encoder_q)
    
    wandb.init(
        # set the wandb project where this run will be logged
        project="dino_clip",
        dir=r"C:\Users\mariu\Desktop",

        # track hyperparameters and run metadata
        config=config
    )

    wandb.watch(model.encoder_q, log="all", log_freq=100)

In [ ]:
dest = os.path.join(r"D:\Dokumente\Studium\Masterprojekt\Gewichte", "dino_clip", DINO_MODEL_NAME.lower() + "_" + CLIP_MODEL_NAME.lower().replace("/", "") + "_" + DATASET.lower() + "_" + datetime.today().strftime('%Y%m%d_%H%M%S'))
os.makedirs(dest)

with open(os.path.join(dest, "constants.json"), "w") as file:
    json.dump(config, file)

def save_model(epoch=0, batch=0, batch_global=0):
    torch.save(model.state_dict(), os.path.join(dest, "state_dict_epoch_" + str(epoch) + "_batch_" + str(batch) + "_batch_global_" + str(batch_global) + ".pt"))

In [ ]:
# Criterion

match CRITERION:
    case "cross_entropy":
        criterion = nn.CrossEntropyLoss().to(device)


# Optimizer

match OPTIMIZER:
    case "sgd":
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()), # model.parameters(),
            LR,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
        )
    case "adam":
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            LR,
            weight_decay=WEIGHT_DECAY,
        )


# Scheduler

scheduler = None

match SCHEDULER:
    # case "step":
    #     scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)
    case "cosine":
        t_max = len(data_loader) * EPOCHS
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=SCHEDULER_MIN_LR)

In [ ]:
def accuracy(output, target, topk=(1,)):
    """Computes the accuracy over the k top predictions for the specified values of k"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.reshape(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

In [ ]:
batch_max = len(data_loader) * EPOCHS
batch_global = 0
EVAL_AFTER = 1024 # Number of batches after which to evaluate on the validation set (= number of batches in an virtual epoch)
SAVE_AFTER = 10240 # 20480

for epoch in range(EPOCHS):
    losses = []
    val_losses = []
    
    tqdm_data_loader = tqdm(data_loader, unit="batch")
    tqdm_data_loader.set_description(f"Epoch {epoch+1}/{EPOCHS}")
    for i, (images, _) in enumerate(tqdm_data_loader):
        images[0] = images[0].to(device)
        images[1] = images[1].to(device)
    
        output, target, _ = model(im_q=images[0], im_k=images[1])
        loss_criterion = criterion(output, target)

        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        alphas = model.get_alphas()

        alpha_reg = torch.zeros((), device=device) # 0.0
        for alpha in alphas.values():
            alpha_reg += (alpha - 1.0) ** 2
        
        loss = loss_criterion + LAMBDA_ALPHA_REG * alpha_reg

        if (i+1) % 10 == 0 and LOG_RUN:
            wandb.log({
                "loss": loss.item(),
                "loss_criterion": loss_criterion.item(),
                "alpha_reg": alpha_reg.item(),
                "moco_acc@1": acc1[0].item(),
                "moco_acc@5": acc5[0].item(),
                "learning_rate": scheduler.get_last_lr()[-1] if scheduler is not None else LR, 
                "epoch": epoch + 1,
                "batch": i + 1,
                "batch_global": batch_global,
                # "clip_alpha": model.encoder_q.clip_alpha.item(),
                # "dino_cls_alpha": model.encoder_q.dino_cls_alpha.item(),
                # "dino_patch_alpha": model.encoder_q.dino_patch_alpha.item()
                **{name: alpha.item() for name, alpha in alphas.items()}
            })

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        postfix = {
            "Loss": loss.item(),
        }

        tqdm_data_loader.set_postfix(postfix)

        # Val
        if (batch_global + 1) % EVAL_AFTER == 0 or (batch_global + 1) == batch_max:
            result = {}

            model.eval()
            with torch.no_grad():
                features = []
                labels = []

                tqdm_data_loader_val = tqdm(data_loader_val, unit="batch")
                tqdm_data_loader_val.set_description(f"Epoch {epoch+1}/{EPOCHS}")
                for i, (images, labels_batch) in enumerate(tqdm_data_loader_val):
                    images[0] = images[0].to(device)
                    images[1] = images[1].to(device)

                    output, target, features_batch = model(im_q=images[0], im_k=images[1])
                    loss = criterion(output, target)

                    val_losses.append(loss.item())
                    features.extend(features_batch.cpu().tolist())
                    labels.extend(labels_batch.tolist())
                features = np.array(features).astype("float32")
                labels = np.array(labels)

                result = evaluate_features(features, labels, EVALUATION_DISTANCE, EVALUATION_K)
            model.train()

            if LOG_RUN:
                wandb.log({
                    "loss_avg": np.mean(losses),
                    "val_loss_avg": np.mean(val_losses),
                    **{f"acc@{k}": result[f"Accuracy@{k}"].item() for k in EVALUATION_K},
                    **{f"per_class_mean_acc@{k}": result["per_class"]["mean"][f"Accuracy@{k}"].item() for k in EVALUATION_K},
                    **{f"per_class_mean_precision@{k}": result["per_class"]["mean"][f"Precision@{k}"].item() for k in EVALUATION_K},
                    **{f"per_class_mean_recall@{k}": result["per_class"]["mean"][f"Recall@{k}"].item() for k in EVALUATION_K}
                })

            losses = []
            val_losses = []

        if (batch_global + 1) % SAVE_AFTER == 0 or (batch_global + 1) == batch_max:
            save_model(epoch=epoch + 1, batch=i + 1, batch_global=batch_global + 1)
        
        if scheduler is not None:
            scheduler.step()
        
        batch_global += 1

In [ ]:
if LOG_RUN:
    wandb.finish()